# 53 · `t1_prior` × hai bộ mã hoá EHR — và `G_ehr` đo bằng **đối chứng khớp tham số**

## Vì sao notebook này tồn tại

`52b` phát hiện một chuyện lật cách đọc bốn notebook trước:

```
t1_col        0.2453
t1_col_zero   0.2468     <- doi chung HANG SO, KHONG EHR, THANG t1_col +0.0015
=> noi dung THUAN cua ColTokBag = -0.0015    (trong khi `arm - mldec` = +0.0084)
```
Tráo nhân quả xác nhận `_zero` thật sự hằng số (`corr 1.000000`, `dmAP +0.0000`), và `t1_col` **có**
đọc EHR (`corr 0.9895`, `dmAP −0.0098`) — nhưng đọc mà không hơn được một khối tham số cùng cỡ.

Xác nhận nội bộ: hai arm hằng số xếp **đúng theo số tham số** — `t1_col_zero` (218 token hằng)
`0.2468` vs `t1_xgb_zero` (41 token hằng) `0.2406`.

⇒ **Mọi `G_ehr` đo bằng `arm − mldec` ở `38c`/`38d`/`44b`/`45` đều PHỒNG.** Notebook này đo lại trên
kiến trúc chính (`t1_prior`) bằng **đối chứng khớp tham số**.

Và `52b` cho `t1_xgb − t1_xgb_zero = +0.0071` trong khi `t1_col − t1_col_zero = −0.0015` ⇒ nghi
**`ColTokBag` là nút thắt**. Ở đây kiểm điều đó trên kiến trúc thật sự dùng.

## 7 arm, **một** backbone forward dùng chung

```
mldec               goc toa do, KHONG EHR, KHONG phim truoc
t1_prior            ColTokBag(217) + Tier1 + Tier1Img       <- KIEN TRUC HIEN TAI
t1_prior_ehrz       token EHR HANG SO      -> doi chung DUNG LUONG cho nhanh EHR
t1_prior_imgz       phim truoc BENH NHAN KHAC -> doi chung cho nhanh ANH (giu tu 45)
t1_prior_xgb        ColTok(40 dac trung XGB tu 52a) + Tier1 + Tier1Img
t1_prior_xgb_ehrz   doi chung HANG SO cho ban XGB
t1_prior_both       ColTokBag(217 + 40)
```

| hiệu | hỏi gì |
|---|---|
| `t1_prior − t1_prior_ehrz` | `ColTokBag` có **nội dung** không (không phải chỉ dung lượng) |
| `t1_prior_xgb − t1_prior_xgb_ehrz` | đặc trưng XGB có nội dung không |
| hiệu của hai cái trên | **bộ mã hoá có phải nút thắt** — câu hỏi chính |
| `t1_prior − t1_prior_imgz` | `G_t1prior`, đo lại trong **cùng** vòng lặp (`45` đo `+0.0029`) |
| `t1_prior_both − max` | hai nguồn **cộng** hay **trùng** |

`t1_prior_xgb` **không** có túi vi khuẩn riêng: 40 đặc trưng XGB đã tóm tắt sẵn toàn bộ EHR **kể cả**
vi khuẩn (`52a`: `XG = [217 cột ; 305 đếm vi khuẩn]`). Thêm túi nữa là đếm hai lần.

## Nhánh và lát cắt

```
huan luyen   TF_       219,785   (train tru lat chon 15% theo subject)
chon E       LAT CHON   39,086   <- E chon o day
bao cao      VAL        39,293   <- MOI so sanh giua cac arm doc o day
theo doi     TEST       78,946   <- CHI infer o 3 epoch cuoi, KHONG tham gia lua chon nao
```

Đây là **nhánh A (phát triển)**. Model chỉ thấy `219,785 / 298,164` ảnh có nhãn — **thiếu 26%** — nên
số test ở đây là **cận dưới**, không phải con số cuối cho bài. Con số đó thuộc nhánh B: huấn luyện
lại trên `train + val` với **cùng `E`** rồi chấm test.

Chấm **3 epoch cuối** chứ không chỉ epoch cuối, để dùng được luật `last-3` của `48` (ensemble epoch:
`+0.0103`, 0 giây GPU) trên test.

## C0 · Config

In [ ]:
import os, json, math, time, random, tarfile, io as _io, re, glob, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r3'      # doi vi Tier1Img da doi (LayerScale) — DUNG resume qua tag cu      # r1 = ban `const` cu (W/b/joint cap phat ma khong dung). DUNG resume qua r1.
RES53   = os.path.join(OUT_DIR, 'stage4_nb53', RUN_TAG)
CKPT    = os.path.join(RES53, 'ckpt')
P_OOF   = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
# TACH theo notebook: hai tien trinh cung `append` mot file qua Drive FUSE se nuot dong
# cua nhau. 53 va 57 chay song song.
P_TOUCH = os.path.join(OUT_DIR, '_test_touches_nb53.jsonl')     # nhat ky MOI lan cham test

# KHONG bao gio va `builtins.open` (de quy vo han + chet worker DataLoader).
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG: mot epoch ~52 phut khong ghi gi la du.'''
    os.makedirs(RES53, exist_ok=True); os.makedirs(CKPT, exist_ok=True); return RES53
ensure_res()
json.dump({'run_tag': RUN_TAG, 'notebook': '53'}, open(os.path.join(RES53, '_run_info.json'), 'w'))
open(os.path.join(CKPT, '_keep'), 'w').write('keep')
LOCAL = '/content/cxrlt_518'

BB_RUN = 'a2_E3'
SEEDS  = (42, 43)             # 2 seed: vua doc duoc bien do, VUA nuoi ensemble cua 48
SEED, SIZE, EPOCHS = 42, 518, 10
BATCH, ACCUM = 64, 4          # HAI anh moi mau (phim truoc) -> giu EFF 256
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
W_FRONTAL, W_LATERAL = 0.7, 0.3
NW = min(8, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0

DO_TEST, N_TEST_EP = True, 3  # cham TEST o 3 epoch CUOI. Xem cong o C10 truoc khi tin con so.

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU -> Runtime > Change runtime type > GPU roi chay lai tu C0.'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

BB_SRC  = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
for p, hd in ((BB_SRC, '38a'), (P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V_ = ((pd.read_parquet(P_VIEW)[['dicom_id','view_group','use']]
         .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
      else ORDER[['dicom_id','view_group']].assign(use=lambda d: d.view_group != 'unknown'))

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL     # CUNG cach cat nhu 35/37/38*/44*/45/46/52*
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')
print(f'backbone `ft` = 38a/{BB_RUN} | seeds {SEEDS} | E={EPOCHS} | BATCH {BATCH}x{ACCUM}')
print(f'TEST: {"BAT" if DO_TEST else "TAT"} — cham {N_TEST_EP} epoch cuoi, CHI de theo doi.')

Mounted at /content/drive
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
377,110 anh | khop 219,785 | chon 39,086 | val 39,293 | test 78,946
backbone `ft` = 38a/a2_E3 | seeds (42, 43) | E=10 | BATCH 64x4
TEST: BAT — cham 3 epoch cuoi, CHI de theo doi.


## C1 · EHR + túi vi khuẩn + đặc trưng XGB + **phim chụp trước** — verbatim `45`/`52b`

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
def _mat(cols):
    m = (E_DF.set_index('study_id')[cols].reindex(_sid)
             .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    return np.nan_to_num(m, nan=0.0)

COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = _mat(COLS)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468; `ColTok` dua thang x_j
# vao token nen score ~ M^2 -> softmax bao hoa -> NaN.
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b
assert (MIDS[MCNT == 0] == PAD_ID).all() and (MIDS[MCNT > 0] != PAD_ID).all()

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all(), f'xgb_oof {XGF.shape}'
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
BOTH_F = np.concatenate([EHR_COL, XGB_F], 1)
print(f'EHR_COL {EHR_COL.shape} | XGB_F {XGB_F.shape} | BOTH_F {BOTH_F.shape} | '
      f'tui vi khuan V={V_MIC-1} dai {MIC_L}')
# GHI CHU: hang TEST cua `xgb_oof` la HANG SO prevalence (52a co y KHONG du doan test). Nen
# `t1_prior_xgb` tren TEST chi nhan mot vector khong doi cho moi anh -> con so test cua HAI arm
# `xgb` KHONG doc duoc. C10 se in canh bao va loai chung khoi bang test.

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5, 'ty le qua thap -> kiem lai ghep study'
assert (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all(), 'phim truoc TRUNG chinh no'

EHR_COL (377110, 217) | XGB_F (377110, 40) | BOTH_F (377110, 257) | tui vi khuan V=305 dai 10
anh co phim truoc: 253,283 / 377,110 (67.2%)  (44a do 67.2%)


## C2 · Ảnh + backbone `ft` đông cứng + `DS` trả **hai** ảnh

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')      # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()   # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
assert _d > 1e-6, f'backbone GIONG HET pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
print(f'BACKBONE `ft`: 38a/{BB_RUN} | lech so voi pretrain {_d:.3f}. CHI lay `bb`.')

class DS(Dataset):
    '''Tra ve CA HAI anh: hien tai va PHIM TRUOC (+ `has`). `ehr_perm` cho phep TRAO NHAN QUA.'''
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return s.tf(Image.merge('RGB', (im, im, im)))
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if pr >= 0 else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), sub=int(SUB[i]), pos=k,
                    y=torch.from_numpy(YM[i]), row=i,
                    erow=(i if s.perm is None else int(s.perm[k])))

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

@torch.no_grad()
def fwd(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return backbone(pixel_values=img).last_hidden_state[:, 1:].float()

_b0 = next(iter(loader(I_VA[:8], eval_tf, False, 8, nw=0)))
N_TOK = fwd(_b0['image'].to(dev)).shape[1]
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'patch {N_TOK} token')

anh local: 0 / 377,110


stage:   0%|          | 0/378 [00:00<?, ?tar/s]

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft`: 38a/a2_E3 | lech so voi pretrain 0.021. CHI lay `bb`.
patch 1369 token


## C3 · Kiến trúc — verbatim `45` (+`ColTokBag` của `52b`)

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    '''`const=True`: MOT chi so, khong phai MIC_L ban sao — tui that trung binh 1.7 vi khuan.'''
    def __init__(s, V, d, const=False):
        super().__init__()
        s.const = const
        s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w):
        if s.const:
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    '''d_in token COT (T_j = b_j + x_j*W_j) + 1 token TONG HOP (MLP nhin CA d_in cot) + 1 token
       BAG vi khuan (tuy chon). Chuoi nay CHUA k1 nhu TRUONG HOP RIENG — attention chi can don het
       khoi luong len token tong hop la tai lap duoc k1.'''
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        # `const` cu thay THANG chuoi token bang mot ma tran hang so, khien `W`/`b`/`joint` duoc
        # CAP PHAT nhung KHONG BAO GIO dung -> chung van bi dem vao `NP_` (nen cong "doi chung bao
        # thu" dat mot cach gia tao) trong khi dung luong THUC te thap hon han arm that.
        # Do la dung lop lech ma `56` phoi bay: doi chung ngheo hon ve VAI TRO thi de sup hon, va
        # moi hieu `arm - doi chung` bi thoi len.
        # Nay doi chung chay DUNG bo may do, chi khac DAU VAO la mot vector HANG SO hoc duoc.
        if const: s.c_in = nn.Parameter(torch.zeros(d_in))
        s.mb = MicroBag(V, d, const) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids=None, mcnt=None):
        if s.const: e = s.c_in.unsqueeze(0).expand(e.shape[0], -1)   # chi DAU VAO la hang so
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
        base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = 1369 PATCH, K/V = token EHR. softmax CHI tren token EHR nen EHR khong the bi 1369 patch
       nhan chim (co che that bai cua 06b H6). `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep
       chieu — ban va 38d, thieu no thi `t1_zero` sup 0.2416 -> 0.0681.'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Phim truoc o muc PATCH. Q = 1369 patch HIEN TAI, K/V = 1369 patch PHIM CU, chay o khong gian
       D_CXR (TRUOC `pin`) => MLDecoder khong phai sua gi.
       KHONG tru thang `patch_cur - patch_prior`: hai phim khong duoc dang ky (tu the, do hit vao),
       o `i` co the la dinh phoi o phim nay va xuong don o phim kia. Cross-attention tu tim tuong ung.'''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        # LAYERSCALE. `pr` zero-init cho dong nhat thuc o buoc 0 nhung KHONG chan duoc toc do
        # lon len: AdamW cap nhat ~lr/buoc/tham so BAT KE gradient nho the nao, nen mot nhanh xuat
        # phat tu 0 lon TUYEN TINH theo buoc x LR. Do `ln_a` chuan hoa `patch - a` (thuong gan 0,
        # phan lon la nhieu) ve chuan don vi, bien do dau ra do TRON VEN `‖pr‖` quyet dinh. Khi no
        # at `patch` thi ML-Decoder doi che do dot ngot -> sup. Da xay ra 3 lan, luon o ep2-4 tuc
        # ngay sau dinh warmup (0.05 x E = nua epoch 1), va `clip` do 0.0% vi gradient VAN binh
        # thuong — van de o BUOC DI, khong o gradient.
        # Nay `pr` khoi tao CHUAN va mot he so LayerScale `1e-4` giu phan du gan 0 luc dau; ca hai
        # deu co gradient that, va he so phai HOC de lon len thay vi lon lên vi quan tinh AdamW.
        s.pr = nn.Linear(d, d)
        s.gamma = nn.Parameter(torch.full((d,), 1e-4))
        s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)      # khong co phim truoc -> mask nhanh di han

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = 1369 patch, co the da bi Tier1 dieu bien.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, w = s.img(q, kv, need_w); return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

kien truc san sang


## C4 · 7 arm + cổng khoá trước

In [ ]:
ARMS = {
  'mldec':             dict(feat=None,   img=False),
  't1_prior':          dict(feat='col',  img=True),
  't1_prior_ehrz':     dict(feat='col',  img=True, const=True),
  't1_prior_imgz':     dict(feat='col',  img=True, shuf=True),
  't1_prior_xgb':      dict(feat='xgb',  img=True, bag=False),   # XGB da tom tat CA vi khuan
  't1_prior_xgb_ehrz': dict(feat='xgb',  img=True, bag=False, const=True),
  't1_prior_both':     dict(feat='both', img=True),
}
FEAT = {'col': EHR_COL, 'xgb': XGB_F, 'both': BOTH_F}

def build_arms(seed=SEED):
    M, E, I = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)      # gieo lai TRUOC TUNG arm -> khoi tao giong het nhau
        if cfg['feat'] is not None:
            E[tag] = ColTokBag(FEAT[cfg['feat']].shape[1], D_ETOK, cfg.get('const', False),
                               V=(V_MIC if cfg.get('bag', True) else None))
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['feat'] is not None)
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
    return M.to(dev), E.to(dev), I.to(dev)

MOD, ETK, IMG = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + ([ETK[t]] if t in ETK else [])
                                  + ([IMG[t]] if t in IMG else []))
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<20} {"tham so":>12}  {"dac trung":>9}  {"tui":>4}')
for t, c in ARMS.items():
    print(f'  {t:<18} {NP_[t]:>12,}  {str(c["feat"]):>9}  {str(c.get("bag", True)):>4}')

_e = {k: torch.randn(4, v.shape[1], device=dev) for k, v in FEAT.items()}
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
_p = torch.randn(4, N_TOK, D_CXR, device=dev); _pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
tok = lambda t: ETK[t](_e[ARMS[t]['feat']], _mi, _mc)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
assert not MOD['mldec'].q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    m = MOD['t1_prior'].eval(); l1, _ = m(_p, tok('t1_prior'))
    _k = m.t1; m.t1 = None; l2, _ = m(_p, tok('t1_prior')); m.t1 = _k
    d1 = (l1 - l2).abs().max().item()
assert d1 < 1e-5, f'Tier1 KHONG zero-init: lech {d1:.2e}'
print(f'\ncong 1: Tier1 zero-init OK (lech {d1:.1e})')
with torch.no_grad():
    for t in IMG:
        # Cong TUONG DOI thay vi tuyet doi: LayerScale cho dong nhat thuc XAP XI (~1e-4), khong
        # bang chinh xac. Va TI LE BIEN DO moi la thu quyet dinh — chinh no vuot 1 la sup.
        _rel = float((IMG[t](_p, _pp, _h1) - _p).norm() / (_p.norm() + 1e-9))
        assert _rel < 1e-3, f'{t}: Tier1Img nhieu {_rel:.2e} luc khoi tao, phai < 1e-3'
print(f'cong 2: Tier1Img zero-init OK cho {len(IMG)} arm')
with torch.no_grad():
    _ti = IMG['t1_prior']; _ti.pr.weight.normal_(0, 0.02)
    _r = float((_ti(_p, _pp*100.0, _h1) - _p).abs().max() / ((_ti(_p, _pp, _h1) - _p).abs().max()+1e-9))
    assert (_ti(_p, _pp, _h0) - _p).abs().max() < 1e-6, 'has=0 KHONG cat duoc nhanh phim truoc'
    nn.init.zeros_(_ti.pr.weight)
assert _r < 3.0, f'phong dai K/V 100x -> bien do ra tang {_r:.1f}x, `ln_a` khong chan duoc'
print(f'cong 3: `ln_a` chan bien do OK ({_r:.2f}x) | has=0 cat sach nhanh OK')
for a, b in (('t1_prior','t1_prior_ehrz'), ('t1_prior_xgb','t1_prior_xgb_ehrz')):
    assert NP_[b] >= NP_[a], f'{b} IT tham so hon {a} -> doi chung khong con bao thu'
    print(f'cong 4: {b} {NP_[b]:,} >= {a} {NP_[a]:,} (+{NP_[b]-NP_[a]:,}) -> `{a} - {b} > 0` BAO THU')
assert NP_['t1_prior'] == NP_['t1_prior_imgz'], 'imgz phai khop tham so TUYET DOI voi t1_prior'
print(f'cong 5: t1_prior = t1_prior_imgz ({NP_["t1_prior"]:,}) — khop TUYET DOI')

arm                       tham so  dac trung   tui
  mldec                 2,517,032       None  True
  t1_prior              7,300,136        col  True
  t1_prior_ehrz         7,300,865        col  True
  t1_prior_imgz         7,300,136        col  True
  t1_prior_xgb          7,019,560        xgb  False
  t1_prior_xgb_ehrz     7,019,600        xgb  False
  t1_prior_both         7,330,856       both  True

cong 1: Tier1 zero-init OK (lech 0.0e+00)
cong 2: Tier1Img zero-init OK cho 6 arm
cong 3: `ln_a` chan bien do OK (1.00x) | has=0 cat sach nhanh OK
cong 4: t1_prior_ehrz 7,300,865 >= t1_prior 7,300,136 (+729) -> `t1_prior - t1_prior_ehrz > 0` BAO THU
cong 4: t1_prior_xgb_ehrz 7,019,600 >= t1_prior_xgb 7,019,560 (+40) -> `t1_prior_xgb - t1_prior_xgb_ehrz > 0` BAO THU
cong 5: t1_prior = t1_prior_imgz (7,300,136) — khop TUYET DOI


## C5 · Chấm điểm — verbatim `45`

In [ ]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
_rs6 = np.random.default_rng(6); _yy = (_rs6.random((3000, C)) < 0.2).astype('float32')
_ss6 = _rs6.random((3000, C)).astype('float32')
_dd6 = float(np.nanmax(np.abs(ap_cols(_yy, _ss6) -
        np.array([average_precision_score(_yy[:, j], _ss6[:, j]) for j in range(C)]))))
assert _dd6 < 1e-6, f'ap_cols lech sklearn {_dd6:.2e}'
print(f'cong 6: ap_cols khop sklearn toi {_dd6:.2e}')

cong 6: ap_cols khop sklearn toi 1.69e-08


## C6 · Huấn luyện — 7 arm, **một** backbone forward

`predict` **shuffle cả khi đánh giá**: `ORDER` sắp theo `subject_id` nên `shuffle=False` có thể cho
một lô toàn một bệnh nhân, và khi đó không hoán vị nào trong lô tìm được bệnh nhân khác. Ghi dự đoán
theo `pos`, không theo thứ tự duyệt.

In [7]:
crit = ASL(gn=4, gp=0).to(dev)
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = {k: torch.from_numpy(v) for k, v in FEAT.items()}
SHUF_SAME = []

def shuf_idx(sb):
    '''Hoan vi TRONG LO sao cho moi hang nhan phim truoc cua BENH NHAN KHAC.
       Ban `prior.roll(1,0)` cho ~83% TRUNG chinh nguoi do luc eval (ORDER sap theo subject_id,
       5.8 anh/BN). Chan cung 32 buoc van con 10.5%. Quet HET B moi sach.'''
    B = sb.shape[0]; idx = torch.arange(B, device=sb.device); p = (idx + 1) % B
    for k in range(2, B):
        bad = sb[p] == sb
        if not bad.any(): break
        p = torch.where(bad, (idx + k) % B, p)
    same = sb[p] == sb
    return p, same, float(same.float().mean())

_rs12 = np.random.default_rng(12)
_rt = float(np.mean([shuf_idx(torch.from_numpy(
    SUB[_rs12.choice(len(SUB), BATCH, replace=False)].astype('int64')).to(dev))[2] for _ in range(20)]))
assert _rt < 0.01, f'lo ngau nhien van {_rt:.2%} trung benh nhan -> doi chung khong dung duoc'
print(f'cong 7: lo ngau nhien {BATCH} mau x 20 lan -> {_rt:.3%} trung benh nhan OK')

def run_arms(M, E, I, patch, erow, prior, has, sb):
    out, pz = {}, None
    for tag, cfg in ARMS.items():
        p = patch
        if cfg['img']:
            if cfg.get('shuf'):
                if pz is None:
                    _pi, _sm, _r = shuf_idx(sb); SHUF_SAME.append(_r)
                    pz = (prior[_pi], has[_pi] * (~_sm).float())   # khong tim duoc -> cat nhanh
                pr, hs = pz
            else: pr, hs = prior, has
            p = I[tag](patch, pr, hs)
        if cfg['feat'] is None: out[tag] = M[tag](p); continue
        e = FEAT_T[cfg['feat']][erow].to(dev)
        out[tag] = M[tag](p, E[tag](e, MIDS_T[erow].to(dev), MCNT_T[erow].to(dev)))
    return out

@torch.no_grad()
def predict(M, E, I, idx, perm=None, bs=48):
    M.eval(); E.eval(); I.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(loader(idx, eval_tf, True, bs, perm, nw=0, drop_last=False), desc='eval', leave=False):
        pt = fwd(b['image'].to(dev, non_blocking=True))
        pp = fwd(b['image_prev'].to(dev, non_blocking=True))
        for t, (lg, _) in run_arms(M, E, I, pt, b['erow'], pp, b['has'].to(dev),
                                   b['sub'].to(dev)).items():
            P[t][b['pos'].numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    for t in ARMS: assert np.abs(P[t]).sum(1).min() > 0, f'{t}: hang khong duoc ghi -> `pos` sai'
    return P

P_LOG = os.path.join(RES53, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL = loader(I_TF, train_tf, True, BATCH)

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    MOD, ETK, IMG = build_arms(seed)
    PARAM_OF = {t: [p for m in ([MOD[t]] + ([ETK[t]] if t in ETK else [])
                                + ([IMG[t]] if t in IMG else []))
                    for p in m.parameters() if p.requires_grad] for t in ARMS}
    N_CLIP = {t: 0 for t in ARMS}; N_SKIP = {t: 0 for t in ARMS}; N_STEP = 0
    rest = [p for m in (MOD, ETK, IMG) for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{'params': [p for p in rest if p.ndim > 1], 'weight_decay': WD},
                             {'params': [p for p in rest if p.ndim <= 1], 'weight_decay': 0.0}], lr=LR)
    steps = math.ceil(len(I_TF) / (BATCH * ACCUM)) * EPOCHS; warm = int(WARMUP * steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})
    P_RES = os.path.join(CKPT, f'{sk}_resume.pt'); ep0 = 0
    if os.path.exists(P_RES):
        _r = torch.load(P_RES, map_location=dev, weights_only=False)
        # Cong RESUME: tap arm phai KHOP. Doi tap arm roi resume se cho RuntimeError dai ba man
        # hinh (da dinh o 52b). Bao ro va neu thang cach xu ly.
        _old = _r.get('arms')
        assert _old is None or list(_old) == list(ARMS), (
            f'checkpoint co {len(_old)} arm {list(_old)}, notebook co {len(ARMS)} arm.\n'
            f'  -> DOI RUN_TAG o C0 (vd "r2") roi chay lai tu C0. KHONG xoa gi.')
        MOD.load_state_dict(_r['mod']); ETK.load_state_dict(_r['etk']); IMG.load_state_dict(_r['img'])
        opt.load_state_dict(_r['opt']); sch.load_state_dict(_r['sch'])
        N_CLIP, N_SKIP, N_STEP, ep0 = _r['n_clip'], _r['n_skip'], _r['n_step'], _r['ep'] + 1
        print(f'{sk}: resume tu epoch {ep0+1}/{EPOCHS}')

    for ep in range(ep0, EPOCHS):
        MOD.train(); ETK.train(); IMG.train()
        t0, tot, nb = time.time(), {t: 0.0 for t in ARMS}, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            pt = fwd(b['image'].to(dev, non_blocking=True)); y = b['y'].to(dev)
            pp = fwd(b['image_prev'].to(dev, non_blocking=True))
            loss, bad = 0.0, []
            for t, (lg, _) in run_arms(MOD, ETK, IMG, pt, b['erow'], pp, b['has'].to(dev),
                                       b['sub'].to(dev)).items():
                if not torch.isfinite(lg).all(): bad.append(t); N_SKIP[t] += 1; continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[t] += float(l_.detach())
            if bad and (sum(N_SKIP.values()) <= 3 or j % 400 == 0):
                print(f'   [{sk} ep{ep+1} buoc {j}] logit khong huu han: {bad}')
            if not torch.is_tensor(loss): opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                # Kiem gradient TRUOC khi buoc: total_norm = nan cho clip_coef = nan va BOI BAN
                # toan bo tham so -> mot buoc hong lam chet vinh vien.
                for t, ps in PARAM_OF.items():
                    gn = torch.nn.utils.clip_grad_norm_(ps, CLIP, error_if_nonfinite=False)
                    if not torch.isfinite(gn):
                        for p in ps:
                            if p.grad is not None: p.grad.zero_()
                        N_SKIP[t] += 1
                    elif float(gn) > CLIP: N_CLIP[t] += 1
                N_STEP += 1; opt.step(); sch.step(); opt.zero_grad(set_to_none=True)

        PV = predict(MOD, ETK, IMG, I_VA); PS = predict(MOD, ETK, IMG, I_SL)
        ensure_res()
        for t, p in PV.items(): np.save(os.path.join(RES53, f'p_{t}_{sk}_ep{ep+1}.npy'), p)
        for t, p in PS.items(): np.save(os.path.join(RES53, f'psl_{t}_{sk}_ep{ep+1}.npy'), p)
        if DO_TEST and ep >= EPOCHS - N_TEST_EP:
            PT = predict(MOD, ETK, IMG, I_TE); ensure_res()
            for t, p in PT.items(): np.save(os.path.join(RES53, f'pte_{t}_{sk}_ep{ep+1}.npy'), p)
            with open(P_TOUCH, 'a') as _f:
                _f.write(json.dumps(dict(nb='53', run=RUN_TAG, seed=seed, ep=ep+1,
                                         n=len(I_TE), arms=list(ARMS), muc_dich='theo doi',
                                         t=datetime.datetime.now().isoformat(timespec='seconds'))) + '\n')
            print(f'   [TEST] da infer {len(I_TE):,} anh, ghi pte_*_ep{ep+1}.npy va _test_touches.jsonl')
        _e2 = {}
        for t, p in PV.items():
            ap = ap_cols(YM[VA], p)
            _e2[t] = dict(mAP_anh=float(np.nanmean(ap)), **tiers(ap),
                          sel=float(np.nanmean(ap_cols(YM[SL], PS[t]))))
        LOG[sk][str(ep)] = _e2; json.dump(LOG, open(P_LOG, 'w'))
        torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                        opt=opt.state_dict(), sch=sch.state_dict(), n_clip=N_CLIP, n_skip=N_SKIP,
                        n_step=N_STEP, ep=ep, seed=seed, arms=list(ARMS)), P_RES)
        print(f'{sk} ep{ep+1:>2} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph | loss ' +
              ' '.join(f'{t.replace("t1_prior","tp")}:{tot[t]/max(nb,1):.3f}' for t in ARMS))
        print('   VAL      ' + ' '.join(f'{t.replace("t1_prior","tp"):>10}:{_e2[t]["mAP_anh"]:.4f}'
                                        for t in ARMS))
        print('   LAT CHON ' + ' '.join(f'{t.replace("t1_prior","tp"):>10}:{_e2[t]["sel"]:.4f}'
                                        for t in ARMS))
        print('   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`')
        _g = lambda a, b: _e2[a]['mAP_anh'] - _e2[b]['mAP_anh']
        print(f'   NOI DUNG col {_g("t1_prior","t1_prior_ehrz"):+.4f} | '
              f'xgb {_g("t1_prior_xgb","t1_prior_xgb_ehrz"):+.4f} | '
              f'hieu {_g("t1_prior_xgb","t1_prior_xgb_ehrz")-_g("t1_prior","t1_prior_ehrz"):+.4f}'
              '  <- CAU HOI CHINH')
        print(f'   G_t1prior {_g("t1_prior","t1_prior_imgz"):+.4f} (45 do +0.0029) | '
              f'both - max {_e2["t1_prior_both"]["mAP_anh"] - max(_e2["t1_prior"]["mAP_anh"], _e2["t1_prior_xgb"]["mAP_anh"]):+.4f}')
        _ss = float(np.mean(SHUF_SAME)) if SHUF_SAME else 0.0
        assert _ss < 0.01, f'doi chung TRUNG benh nhan {_ss:.1%} -> `imgz` khong phai doi chung'
        SHUF_SAME.clear()
        _f = {t: N_CLIP[t]/max(N_STEP,1) for t in ARMS}
        if max(_f.values()) > 0.01:
            print('   clip: ' + ' '.join(f'{t}:{v:.0%}' for t, v in _f.items() if v > 0.01))
        if max(N_SKIP.values()) > 0:
            print('   BUOC BI BO: ' + ' '.join(f'{t}:{v}' for t, v in N_SKIP.items() if v > 0))
    ensure_res()
    torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                    seed=seed, arms=list(ARMS)), os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong')
print('tat ca seed xong')

cong 7: lo ngau nhien 64 mau x 20 lan -> 0.000% trung benh nhan OK


s42 ep1/10:   0%|          | 0/3434 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/815 [00:00<?, ?it/s]

s42 ep 1 | 52 img/s | 69.9 ph | loss mldec:0.043 tp:0.043 tp_ehrz:0.043 tp_imgz:0.043 tp_xgb:0.042 tp_xgb_ehrz:0.042 tp_both:0.041
   VAL           mldec:0.2368         tp:0.2436    tp_ehrz:0.2423    tp_imgz:0.2398     tp_xgb:0.2445 tp_xgb_ehrz:0.2406    tp_both:0.2443
   LAT CHON      mldec:0.2274         tp:0.2335    tp_ehrz:0.2334    tp_imgz:0.2316     tp_xgb:0.2336 tp_xgb_ehrz:0.2325    tp_both:0.2346
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   NOI DUNG col +0.0013 | xgb +0.0039 | hieu +0.0026  <- CAU HOI CHINH
   G_t1prior +0.0037 (45 do +0.0029) | both - max -0.0002


...

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/815 [00:00<?, ?it/s]

s43 ep 6 | 53 img/s | 69.6 ph | loss mldec:0.037 tp:0.037 tp_ehrz:0.037 tp_imgz:0.037 tp_xgb:0.037 tp_xgb_ehrz:0.037 tp_both:0.037
   VAL           mldec:0.2426         tp:0.2503    tp_ehrz:0.2477    tp_imgz:0.2479     tp_xgb:0.2513 tp_xgb_ehrz:0.2476    tp_both:0.2528
   LAT CHON      mldec:0.2367         tp:0.2431    tp_ehrz:0.2383    tp_imgz:0.2404     tp_xgb:0.2465 tp_xgb_ehrz:0.2401    tp_both:0.2444
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   NOI DUNG col +0.0026 | xgb +0.0037 | hieu +0.0011  <- CAU HOI CHINH
   G_t1prior +0.0024 (45 do +0.0029) | both - max +0.0015


s43 ep7/10:   0%|          | 0/3434 [00:00<?, ?it/s]

KeyboardInterrupt: 

## C7 · Kết quả — **nội dung thuần**, không phải `arm − mldec`

In [ ]:
LOG = json.load(open(P_LOG)); EPk = str(EPOCHS - 1)
R = pd.DataFrame([dict(seed=s, arm=t, **v) for s in SEEDS if EPk in LOG.get(f's{s}', {})
                  for t, v in LOG[f's{s}'][EPk].items()])
assert len(R), 'chua co ket qua nao'
SD = [s for s in SEEDS if EPk in LOG.get(f's{s}', {})]
piv = R.pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel'],
                    aggfunc='mean').reindex(list(ARMS))
piv['bien_do'] = R.groupby('arm').mAP_anh.agg(lambda s: s.max()-s.min()).reindex(list(ARMS))
piv['tham_so'] = [NP_[t] for t in ARMS]
print(f'=== TUYET DOI (TB qua {len(SD)} seed) ===')
print(piv.to_string(float_format=lambda v: f'{v:.4f}'))
NOISE = float(np.nanmax(piv.bien_do)) if len(SD) > 1 else float('nan')
print(f'\nbien do seed lon nhat: ' + (f'{NOISE:.4f}' if len(SD) > 1 else 'CHUA DOC DUOC (1 seed)'))

g = lambda a, s: float(R[(R.arm == a) & (R.seed == s)].mAP_anh.iloc[0])
def d(a, b, ngh):
    v = [g(a, s) - g(b, s) for s in SD]
    ok = len(v) > 1 and (all(x > 0 for x in v) or all(x < 0 for x in v))
    bd = (max(v) - min(v)) if len(v) > 1 else float('nan')
    dat = '---' if len(v) < 2 else ('DAT' if (ok and abs(np.mean(v)) > bd) else '---')
    print(f'  {a:>17} - {b:<17} {np.mean(v):+.4f}  (bien do {bd:.4f}, {dat})   {ngh}')
    return float(np.mean(v))

print('\n=== NOI DUNG THUAN (arm - doi chung KHOP THAM SO) — DAY moi la so doc duoc ===')
Dc = d('t1_prior', 't1_prior_ehrz', 'ColTokBag co NOI DUNG khong')
Dx = d('t1_prior_xgb', 't1_prior_xgb_ehrz', 'dac trung XGB co NOI DUNG khong')
print(f'\n  CAU HOI CHINH  noi dung(xgb) - noi dung(col) = {Dx-Dc:+.4f}')
print('    > +0.005 -> `ColTokBag` LA nut that (52b tren mldec: -0.0015 vs +0.0071)')
print('    ~ 0      -> KHONG phai; tran nam o NGUON NHAN (nhan trich tu report cua chinh phim)')
print('\n=== CAC HIEU KHAC ===')
d('t1_prior', 't1_prior_imgz', 'G_t1prior — nhanh ANH (45 do +0.0029)')
d('t1_prior_both', 't1_prior', 'them 40 cot XGB vao ColTokBag')
d('t1_prior_xgb', 't1_prior', 'thay ColTokBag BANG XGB')
print('\n=== so voi `arm - mldec` (PHONG — 52b da chung minh) ===')
for t in ARMS:
    if t == 'mldec': continue
    print(f'  {t:<20} - mldec  {np.mean([g(t,s)-g("mldec",s) for s in SD]):+.4f}')
print('  ^ 52b: t1_col - mldec = +0.0084 nhung t1_col - t1_col_zero = -0.0015. Doc cot TREN.')
ensure_res(); R.to_csv(os.path.join(RES53, 'main_table.csv'), index=False)

## C8 · Tráo nhân quả + nhóm nhãn

In [ ]:
ck = torch.load(os.path.join(CKPT, f's{SD[0]}.pt'), map_location=dev, weights_only=False)
MOD.load_state_dict(ck['mod']); ETK.load_state_dict(ck['etk']); IMG.load_state_dict(ck['img'])
perm = np.random.default_rng(0).permutation(len(I_VA))
P0 = predict(MOD, ETK, IMG, I_VA); P1 = predict(MOD, ETK, IMG, I_VA, perm=I_VA[perm])
lgt = lambda p: np.log(p.clip(1e-6, 1-1e-6) / (1 - p.clip(1e-6, 1-1e-6)))
print(f'=== TRAO EHR SANG BENH NHAN KHAC (s{SD[0]}) — NHAN giu nguyen ===')
print('  Doc HAI cot: corr mot minh QUA CHAT (44b tung co corr = 1.000000 ma mat 0.0056 mAP).')
for t in ARMS:
    if ARMS[t]['feat'] is None: continue
    r = float(np.corrcoef(lgt(P0[t]).ravel(), lgt(P1[t]).ravel())[0, 1])
    dd = float(np.nanmean(ap_cols(YM[VA], P1[t])) - np.nanmean(ap_cols(YM[VA], P0[t])))
    v = ('EHR co tac dong that' if dd <= -0.002 else
         'EHR KHONG duoc dung' if r > 0.9999 and dd > -0.0005 else 'vung xam')
    print(f'  {t:<20} corr {r:.6f} | mAP khi trao {dd:+.4f}   {v}')
    if ARMS[t].get('const'):
        assert r > 1 - 1e-9 and abs(dd) < 1e-9, f'{t} la doi chung HANG SO ma trao van doi -> ro ri'
print('  moc 06b: corr = 0.999934 (H6)')

INF = [l for l in LAB40 if re.search(r'pneumon|tubercul|infect|consolidat|abscess|empyema', l, re.I)]
STR_ = [l for l in LAB40 if re.search(
    r'pneumothorax|hernia|fracture|mass|nodule|calcif|scoliosis|support device', l, re.I)]
I_INF = np.array([LAB40.index(l) for l in INF]); I_STR = np.array([LAB40.index(l) for l in STR_])
sk = f's{SD[0]}'; _r = []
for t in ARMS:
    f = os.path.join(RES53, f'p_{t}_{sk}_ep{EPOCHS}.npy')
    if not os.path.exists(f): continue
    a = ap_cols(YM[VA], np.load(f))
    _r.append(dict(arm=t, macro=float(np.nanmean(a)), nhiem_trung=float(np.nanmean(a[I_INF])),
                   cau_truc=float(np.nanmean(a[I_STR])), tail=float(np.nanmean(a[TIER['tail']]))))
T = pd.DataFrame(_r).set_index('arm')
for c in ('macro','nhiem_trung','cau_truc','tail'): T[f'd_{c}'] = T[c] - T.loc['mldec', c]
print(f'\nnhiem trung ({len(INF)}) | cau truc ({len(STR_)})')
print(T.to_string(float_format=lambda v: f'{v:+.4f}'))
print('\n  EHR an nhom NHIEM TRUNG · PHIM TRUOC an nhom CAU TRUC (44a). Hai nguon HAI nhom.')

## C9 · TEST — **chỉ theo dõi**, ba cổng trước khi tin con số

Test **không** tham gia bất kỳ lựa chọn nào. `E` đã chọn trên `LAT CHON`, mọi so sánh arm đọc ở `VAL`.

Ba cổng — nếu bộ nhãn test bị giữ lại thì `YM[TEST]` có thể toàn 0 và mAP in ra là **rác trông hợp lý**:

```
1  so nhan co >=1 ca duong tren test  >= 35 / 40
2  prevalence(test) / prevalence(val)  trong [0.5, 2.0]
3  hai arm `xgb` BI LOAI khoi bang test — `52a` co y KHONG du doan test, hang test cua
   `xgb_oof` la HANG SO prevalence, nen dac trung cua chung tren test la vo nghia
```

In [ ]:
if not DO_TEST:
    print('DO_TEST = False -> bo qua')
else:
    _pv_te, _pv_va = YM[TE].mean(0), YM[VA].mean(0)
    _nlab = int((YM[TE].sum(0) > 0).sum()); _rat = float(np.nanmean(_pv_te / (_pv_va + 1e-9)))
    print(f'cong 1: {_nlab}/{C} nhan co >=1 ca duong tren test')
    print(f'cong 2: prevalence(test)/prevalence(val) TB = {_rat:.3f}')
    assert _nlab >= 35, (f'chi {_nlab}/{C} nhan co ca duong tren test -> nhan test co ve BI GIU LAI. '
                         'Moi so test deu vo nghia. Dat DO_TEST = False.')
    assert 0.5 <= _rat <= 2.0, f'prevalence lech {_rat:.2f}x -> kiem lai nhan test'
    _SKIP = [t for t in ARMS if ARMS[t]['feat'] in ('xgb',)]
    print(f'cong 3: LOAI khoi bang test (dac trung XGB tren test la HANG SO): {_SKIP}')
    _rows = []
    for s in SD:
        for t in ARMS:
            if t in _SKIP: continue
            fs = [os.path.join(RES53, f'pte_{t}_s{s}_ep{e}.npy')
                  for e in range(EPOCHS - N_TEST_EP + 1, EPOCHS + 1)]
            fs = [f for f in fs if os.path.exists(f)]
            if not fs: continue
            a_last = ap_cols(YM[TE], np.load(fs[-1]))
            a_ens  = ap_cols(YM[TE], np.mean([np.load(f) for f in fs], 0))   # luat last-3 cua 48
            _rows.append(dict(seed=s, arm=t, test_ep_cuoi=float(np.nanmean(a_last)),
                              test_last3=float(np.nanmean(a_ens)),
                              test_tail=float(np.nanmean(a_ens[TIER['tail']]))))
    TT = pd.DataFrame(_rows)
    if len(TT):
        pt = TT.pivot_table(index='arm', values=['test_ep_cuoi','test_last3','test_tail'],
                            aggfunc='mean')
        pt['val'] = [piv.loc[a, 'mAP_anh'] for a in pt.index]
        pt['test_last3 - val'] = pt.test_last3 - pt.val
        print('\n' + pt.to_string(float_format=lambda v: f'{v:.4f}'))
        ensure_res(); TT.to_csv(os.path.join(RES53, 'test_table.csv'), index=False)
    print('\n  !! CANH BAO DOC SO: model nay chi thay 219,785 / 298,164 anh co nhan (THIEU 26%).')
    print('     Con so test o day la CAN DUOI cua cau hinh, KHONG phai con so cuoi cho bai.')
    print('     Con so do thuoc NHANH B: huan luyen lai tren train+val voi CUNG `E` roi cham test.')
    print(f'     Moi lan cham da ghi vao {P_TOUCH}.')

## C10 · Ghi `REPORT_STAGE4_NB53.md`

In [ ]:
L = []; w = L.append
w('# REPORT · `nb53` — `t1_prior` × hai bộ mã hoá EHR, đo bằng đối chứng khớp tham số'); w('')
w(f'`{RUN_TAG}` · backbone `ft` = `38a/{BB_RUN}` · `asl40` · `E={EPOCHS}` · **epoch cuối** · '
  f'seeds {list(SD)} · {len(ARMS)} arm **một backbone forward dùng chung** · val {int(VA.sum()):,} ảnh.')
w('')
w('| arm | tham số | mAP ảnh | biên độ | tail | nhiễm trùng |'); w('|---|---|---|---|---|---|')
for t in ARMS:
    if t not in piv.index: continue
    r = piv.loc[t]
    w(f'| `{t}` | {NP_[t]:,} | {r["mAP_anh"]:.4f} | {r["bien_do"]:.4f} | {r["tail"]:.4f} | '
      + (f'{T.loc[t, "nhiem_trung"]:.4f}' if t in T.index else '—') + ' |')
w(''); w('```')
w(f'noi dung(col) = t1_prior     - t1_prior_ehrz      {Dc:+.4f}')
w(f'noi dung(xgb) = t1_prior_xgb - t1_prior_xgb_ehrz  {Dx:+.4f}')
w(f'CAU HOI CHINH  noi dung(xgb) - noi dung(col)      {Dx-Dc:+.4f}')
w('```'); w('')
w('**Chỉ đọc cột NỘI DUNG THUẦN.** `52b` đo `t1_col − mldec = +0.0084` nhưng '
  '`t1_col − t1_col_zero = −0.0015`: đối chứng hằng số **thắng** arm thật ⇒ mọi `G_ehr` đo bằng '
  '`arm − mldec` ở `38c`/`38d`/`44b`/`45` đều **phồng**.'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **Backbone đông cứng** ⇒ EHR chỉ điều biến biểu diễn đã xong. Deep conditioning (`39`, `B2d`) '
  'là cơ chế duy nhất chưa ai đóng.')
w('- **`p_uni` chưa chạm** — tỉ lệ mẫu uni-modal là thay đổi lấy mẫu của cả vòng lặp, không làm '
  'được thành một arm. MedFuse đo `+0.015` từ đúng đòn bẩy này.')
w(f'- **Số test là CẬN DƯỚI**: model chỉ thấy {int(TF_.sum()):,} / {int((TR|VA).sum()):,} ảnh có '
  'nhãn. Test **không** tham gia lựa chọn nào; mọi lần chạm ghi ở `_test_touches.jsonl`.')
w('- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh gần như đủ theo định '
  'nghĩa (`34`: nhãn tự động gọi thừa `Pneumonia` **5×**).')
TXT = '\n'.join(L); ensure_res()
open(os.path.join(RES53, 'REPORT_STAGE4_NB53.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)